In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml

# 1. Carregamento do dataset MNIST
print("Baixando o dataset MNIST (mnist_784)...")
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
X, y = mnist.data, mnist.target.astype(int)

# 2. Exibição da dimensionalidade das matrizes X e y
print(f"\nDimensões de X (matriz de características): {X.shape}")
print(f"Dimensões de y (vetor de rótulos): {y.shape}")

# 3. Comprovação do balanceamento das classes (dígitos de 0 a 9)
df_y = pd.Series(y)
print("\nDistribuição de frequência das classes (dígitos 0 a 9):")
print(df_y.value_counts().sort_index())

Baixando o dataset MNIST (mnist_784)...

Dimensões de X (matriz de características): (70000, 784)
Dimensões de y (vetor de rótulos): (70000,)

Distribuição de frequência das classes (dígitos 0 a 9):
0    6903
1    7877
2    6990
3    7141
4    6824
5    6313
6    6876
7    7293
8    6825
9    6958
Name: count, dtype: int64


In [4]:
from sklearn.model_selection import train_test_split

# 1. Divisão Estratificada: 80% (Treino + Validação) e 20% (Teste)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 2. Subdivisão da base de treino: 70% Treino e 10% Validação (do total original)
# 0.125 de 80% equivale a 10% do total original (70% treino total)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.125, random_state=42, stratify=y_train_val
)

# 3. Normalização / Escalonamento dos pixels para a escala [0.0, 1.0]
X_train = X_train / 255.0
X_val = X_val / 255.0
X_test = X_test / 255.0

# 4. Exibição das dimensões dos conjuntos resultantes
print(f"Conjunto de Treino:    {X_train.shape} amostras ({len(X_train)/len(X)*100:.0f}%)")
print(f"Conjunto de Validação: {X_val.shape} amostras ({len(X_val)/len(X)*100:.0f}%)")
print(f"Conjunto de Teste:     {X_test.shape} amostras ({len(X_test)/len(X)*100:.0f}%)")

Conjunto de Treino:    (49000, 784) amostras (70%)
Conjunto de Validação: (7000, 784) amostras (10%)
Conjunto de Teste:     (14000, 784) amostras (20%)


### Justificativa Técnica do Pré-processamento

* **Importância da Estratificação (`stratify=y`)**: Mantém a exata proporção da distribuição dos dígitos (0 a 9) em todas as divisões (treino, validação e teste), evitando viés amostral no treinamento e garantindo uma avaliação justa de todas as classes.
* **Impacto da Normalização ([0.0, 1.0])**: Escalonar os valores dos pixels da faixa original de 0–255 para 0.0–1.0 é essencial para acelerar a convergência de otimizadores baseados em gradiente (como em Redes Neurais/MLP e Regressão Logística) e evitar o domínio desproporcional de características em algoritmos baseados em distâncias métricas (como KNN e SVM).

## Fase 3: Implementação e Treinamento dos 3 Modelos

Para a comparação de desempenho, foram selecionadas três arquiteturas distintas:
1. **Random Forest**: Algoritmo baseado em ensemble de árvores de decisão.
2. **Rede Neural MLP (Multilayer Perceptron)**: Modelo de aprendizado profundo (Deep Learning).
3. **Regressão Logística Multinomial**: Classificador linear de referência.


In [5]:
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression

# Dicionário para armazenar os modelos treinados e os tempos de execução
models = {}
execution_times = {}

# ---------------------------------------------------------
# Modelo 1: Random Forest
# Hiperparâmetros ajustados: n_estimators=100, max_depth=15
# ---------------------------------------------------------
print("Treinando Modelo 1: Random Forest...")
start_time = time.time()
rf_model = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
t_rf = time.time() - start_time

models['Random Forest'] = rf_model
execution_times['Random Forest'] = t_rf
print(f"Random Forest concluído em {t_rf:.2f} segundos.")

# ---------------------------------------------------------
# Modelo 2: MLPClassifier (Rede Neural)
# Hiperparâmetros ajustados: hidden_layer_sizes=(128, 64), max_iter=20
# ---------------------------------------------------------
print("\nTreinando Modelo 2: Rede Neural (MLP)...")
start_time = time.time()
mlp_model = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=20, random_state=42)
mlp_model.fit(X_train, y_train)
t_mlp = time.time() - start_time

models['MLP'] = mlp_model
execution_times['MLP'] = t_mlp
print(f"MLP concluído em {t_mlp:.2f} segundos.")

# ---------------------------------------------------------
# Modelo 3: Regressão Logística Multinomial
# Hiperparâmetros ajustados: C=1.0, max_iter=150
# ---------------------------------------------------------
print("\nTreinando Modelo 3: Regressão Logística...")
start_time = time.time()
logreg_model = LogisticRegression(C=1.0, max_iter=150, random_state=42)
logreg_model.fit(X_train, y_train)
t_logreg = time.time() - start_time

models['Regressão Logística'] = logreg_model
execution_times['Regressão Logística'] = t_logreg
print(f"Regressão Logística concluída em {t_logreg:.2f} segundos.")

Treinando Modelo 1: Random Forest...
Random Forest concluído em 21.78 segundos.

Treinando Modelo 2: Rede Neural (MLP)...


c:\Users\Richard Campos\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


MLP concluído em 68.08 segundos.

Treinando Modelo 3: Regressão Logística...
Regressão Logística concluída em 423.39 segundos.


c:\Users\Richard Campos\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 150 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=150).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### Justificativa Técnica dos Hiperparâmetros Ajustados

1. **Random Forest**:
   * `n_estimators=100`: Define a quantidade de árvores de decisão no conjunto (*ensemble*), garantindo estabilidade nas previsões e reduzindo a variância.
   * `max_depth=15`: Limita a profundidade máxima de cada árvore para evitar *overfitting* e controlar a complexidade computacional.

2. **Perceptron Multicamadas (MLP)**:
   * `hidden_layer_sizes=(128, 64)`: Estrutura a arquitetura com duas camadas ocultas contendo 128 e 64 neurônios, permitindo a extração de características não lineares complexas das imagens.
   * `max_iter=20`: Define o número máximo de épocas de treinamento para o algoritmo de retropropagação (*backpropagation*).

3. **Regressão Logística**:
   * `C=1.0`: Parâmetro de regularização inversa que controla a penalização de pesos, prevenindo *overfitting*.
   * `max_iter=150`: Garante número suficiente de iterações para a convergência do otimizador numérico.